# Module 2: Bias Detection
**CAP6412 — Bias & Safety Auditor for T2I Models**

CLIP zero-shot classification for demographic attributes (gender, age, race) across generated images.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

from src.bias_detector import BiasDetector

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
detector = BiasDetector(clip_model='ViT-B/32', device=device)
print('Bias detector loaded.')

In [ ]:
# Classify a single image
from PIL import Image
import glob

test_images = list(Path('../images').rglob('*.png'))[:1]
if test_images:
    result = detector.classify_image(str(test_images[0]))
    print(f'Image: {test_images[0].name}')
    for attr in ['gender', 'age', 'race']:
        print(f'  {attr}: {result[f"{attr}_pred"]} ({result[f"{attr}_confidence"]:.1%})')

In [ ]:
# Run full analysis
Path('../results').mkdir(exist_ok=True)
df, summary = detector.analyse_all('../images', '../prompts.csv')

if not df.empty:
    df.to_csv('../results/bias_results.csv', index=False)
    with open('../results/bias_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    print(f'Saved {len(df)} records.')
    df.head()

In [ ]:
# Load and explore results
df = pd.read_csv('../results/bias_results.csv')
print(df[['category', 'prompt', 'gender_pred', 'race_pred', 'age_pred']].head(15))

In [ ]:
# Gender distribution per category
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, cat in zip(axes, df['category'].unique()):
    subset = df[df['category'] == cat]
    counts = subset['gender_pred'].value_counts()
    counts.plot(kind='bar', ax=ax, color=['#3949AB', '#E91E63', '#9E9E9E'])
    ax.set_title(f'Gender — {cat}', fontweight='bold')
    ax.set_ylabel('Count')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
plt.suptitle('Gender Distribution by Category', fontsize=13)
plt.tight_layout()
plt.savefig('../results/charts/gender_by_category.png', dpi=150)
plt.show()

In [ ]:
# Race distribution per category
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, cat in zip(axes, df['category'].unique()):
    subset = df[df['category'] == cat]
    counts = subset['race_pred'].value_counts()
    counts.plot(kind='barh', ax=ax, color='#3949AB')
    ax.set_title(f'Race — {cat}', fontweight='bold')
plt.suptitle('Race Distribution by Category', fontsize=13)
plt.tight_layout()
plt.savefig('../results/charts/race_by_category.png', dpi=150)
plt.show()

In [ ]:
# Bias score heatmap across prompts
with open('../results/bias_summary.json') as f:
    summary = json.load(f)

categories = list(summary.keys())
attributes = ['gender', 'age', 'race']
scores = [[summary[cat][attr]['bias_score'] for attr in attributes] for cat in categories]

import numpy as np
fig, ax = plt.subplots(figsize=(7, 3))
im = ax.imshow(scores, cmap='RdYlGn_r', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(attributes)))
ax.set_xticklabels([a.capitalize() for a in attributes])
ax.set_yticks(range(len(categories)))
ax.set_yticklabels([c.capitalize() for c in categories])
for i in range(len(categories)):
    for j in range(len(attributes)):
        ax.text(j, i, f'{scores[i][j]:.2f}', ha='center', va='center', fontsize=10, color='black')
plt.colorbar(im, ax=ax, label='Bias Score (0=fair, 1=biased)')
ax.set_title('Bias Score Heatmap', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/charts/bias_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Top biased occupation prompts
occ = df[df['category'] == 'occupation'].groupby('prompt')['gender_pred'].apply(
    lambda x: (x == 'man').mean()
).sort_values(ascending=False)

plt.figure(figsize=(12, 5))
occ.plot(kind='barh', color='#3949AB')
plt.axvline(x=0.5, color='red', linestyle='--', label='Fair baseline')
plt.xlabel('Proportion predicted Male')
plt.title('Gender Bias per Occupation Prompt', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../results/charts/gender_per_occupation.png', dpi=150)
plt.show()